# Практика застосування SVM-класифікатора. EDA

## EDA датасету PetFinder

### Знайомство з даними

In [1]:
import pandas as pd
import numpy as np
import category_encoders as ce
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import KBinsDiscretizer, StandardScaler
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, confusion_matrix

In [2]:
data = pd.read_csv('../datasets/mod_04_topic_08_petfinder_data.csv.gz')
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 11537 entries, 0 to 11536
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Type           11537 non-null  str  
 1   Age            11537 non-null  int64
 2   Breed1         11537 non-null  str  
 3   Gender         11537 non-null  str  
 4   Color1         11537 non-null  str  
 5   Color2         11537 non-null  str  
 6   MaturitySize   11537 non-null  str  
 7   FurLength      11537 non-null  str  
 8   Vaccinated     11537 non-null  str  
 9   Sterilized     11537 non-null  str  
 10  Health         11537 non-null  str  
 11  Fee            11537 non-null  int64
 12  Description    11528 non-null  str  
 13  PhotoAmt       11537 non-null  int64
 14  AdoptionSpeed  11537 non-null  int64
dtypes: int64(4), str(11)
memory usage: 5.6 MB


In [3]:
data.nunique()

Type                 2
Age                105
Breed1             166
Gender               2
Color1               7
Color2               7
MaturitySize         3
FurLength            3
Vaccinated           3
Sterilized           3
Health               3
Fee                 65
Description      10691
PhotoAmt            31
AdoptionSpeed        5
dtype: int64

### Адаптація ознак і цільової змінної

In [4]:
data['Description'].head()

0    Nibble is a 3+ month old ball of cuteness. He ...
1    I just found it alone yesterday near my apartm...
2    Their pregnant mother was dumped by her irresp...
3    Good guard dog, very alert, active, obedience ...
4    This handsome yet cute boy is up for adoption....
Name: Description, dtype: str

In [5]:
data = data.drop('Description', axis=1)

In [6]:
data['AdoptionSpeed'].value_counts().sort_index()

AdoptionSpeed
0     329
1    2432
2    3153
3    2543
4    3080
Name: count, dtype: int64

In [7]:
data['AdoptionSpeed'] = np.where(data['AdoptionSpeed'] == 4, 0, 1)
data['AdoptionSpeed'].value_counts()

AdoptionSpeed
1    8457
0    3080
Name: count, dtype: int64

In [8]:
data['Fee'] = data['Fee'].astype(bool).astype(int).astype(str)

# Практика застосування SVM-класифікатора. Підготовка й обробка даних

### Розбиття на тренувальну і тестову вибірки

In [9]:
X_train, X_test, y_train, y_test = (
    train_test_split(
        data.drop('AdoptionSpeed', axis=1),
        data['AdoptionSpeed'],
        test_size=0.2,
        random_state=42))

In [10]:
num_cols = X_train.select_dtypes(exclude='str').columns

kbins = KBinsDiscretizer(encode='ordinal').fit(X_train[num_cols])

X_train[num_cols] = (kbins
                     .transform(
                         X_train[num_cols])
                     .astype(int)
                     .astype(str))

X_test[num_cols] = (kbins
                    .transform(
                        X_test[num_cols])
                    .astype(int)
                    .astype(str))

### Кодування категоріальних змінних

In [11]:
encoder = ce.TargetEncoder()

X_train = encoder.fit_transform(X_train, y_train)
X_test = encoder.transform(X_test)

X_train.head()

,Type,Age,Breed1,Gender,Color1,Color2,MaturitySize,FurLength,Vaccinated,Sterilized,Health,Fee,PhotoAmt
3527,0.719879,0.856507,0.669992,0.760396,0.712899,0.714331,0.725716,0.715193,0.797233,0.808532,0.739566,0.742222,0.719681
6377,0.719879,0.622047,0.669992,0.716901,0.712899,0.714331,0.766751,0.715193,0.688564,0.613491,0.739566,0.742222,0.719681
8256,0.719879,0.893978,0.881118,0.716901,0.733775,0.712644,0.725716,0.715193,0.797233,0.808532,0.739566,0.742222,0.683806
916,0.719879,0.749135,0.669992,0.716901,0.712899,0.755591,0.766751,0.715193,0.689878,0.613491,0.739566,0.742222,0.719681
5972,0.719879,0.622047,0.881118,0.760396,0.712899,0.714331,0.741463,0.715193,0.689878,0.599174,0.739566,0.742222,0.806862


### Нормалізація змінних

In [12]:
scaler = StandardScaler().set_output(transform='pandas')

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Практика застосування SVM-класифікатора. Навчання й оцінка моделі

### Побудова моделі

In [13]:
clf = CalibratedClassifierCV(
    SVC(class_weight='balanced',
        kernel='poly'),
    method='isotonic',
    ensemble=False)

clf.fit(X_train, y_train)

,"estimator estimator: estimator instance, default=NoneThe classifier whose output need to be calibrated to provide moreaccurate `predict_proba` outputs. The default classifier isa :class:`~sklearn.svm.LinearSVC`... versionadded:: 1.2",SVC(class_wei...kernel='poly')
,"method method: {'sigmoid', 'isotonic', 'temperature'}, default='sigmoid'The method to use for calibration. Can be:- 'sigmoid', which corresponds to Platt's method (i.e. a binary logistic regression model).- 'isotonic', which is a non-parametric approach.- 'temperature', temperature scaling.Sigmoid and isotonic calibration methods natively support only binaryclassifiers and extend to multi-class classification using a One-vs-Rest (OvR)strategy with post-hoc renormalization, i.e., adjusting the probabilities aftercalibration to ensure they sum up to 1.In contrast, temperature scaling naturally supports multi-class calibration byapplying `softmax(classifier_logits/T)` with a value of `T` (temperature)that optimizes the log loss.For very uncalibrated classifiers on very imbalanced datasets, sigmoidcalibration might be preferred because it fits an additional interceptparameter. This helps shift decision boundaries appropriately when theclassifier being calibrated is biased towards the majority class.Isotonic calibration is not recommended when the number of calibration samplesis too low ``(≪1000)`` since it then tends to overfit... versionchanged:: 1.8 Added option 'temperature'.",'isotonic'
,"ensemble ensemble: bool, or ""auto"", default=""auto""Determines how the calibrator is fitted.""auto"" will use `False` if the `estimator` is a:class:`~sklearn.frozen.FrozenEstimator`, and `True` otherwise.If `True`, the `estimator` is fitted using training data, andcalibrated using testing data, for each `cv` fold. The final estimatoris an ensemble of `n_cv` fitted classifier and calibrator pairs, where`n_cv` is the number of cross-validation folds. The output is theaverage predicted probabilities of all pairs.If `False`, `cv` is used to compute unbiased predictions, via:func:`~sklearn.model_selection.cross_val_predict`, which are thenused for calibration. At prediction time, the classifier used is the`estimator` trained on all the data.Note that this method is also internally implemented in:mod:`sklearn.svm` estimators with the `probabilities=True` parameter... versionadded:: 0.24.. versionchanged:: 1.6 `""auto""` option is added and is the default.",False
,"cv cv: int, cross-validation generator, or iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- integer, to specify the number of folds,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used. If ``y`` isneither binary nor multiclass, :class:`~sklearn.model_selection.KFold`is used.Refer to the :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors.Base estimator clones are fitted in parallel across cross-validationiterations.See :term:`Glossary <n_jobs>` for more details... versionadded:: 0.24",None
Name,Type,Value
"calibrated_classifiers_ calibrated_classifiers_: list (len() equal to cv or 1 if `ensemble=False`)The list of classifier and calibrator pairs.- When `ensemble=True`, `n_cv` fitted `estimator` and calibrator pairs. `n_cv` is the number of cross-validation folds.- When `ensemble=False`, the `estimator`, fitted on all the data, and fitted calibrator... versionchanged:: 0.24 Single calibrated classifier case when `ensemble=False`.",list,[<sklearn.cali...0016B4127B4D0>]
"

In [14]:
preds = clf.predict(X_test)

confusion_matrix(y_test, preds)

array([[ 246,  397],
       [ 190, 1475]])

In [15]:
print(f'Model accuracy is: {accuracy_score(y_test, preds):.1%}')

Model accuracy is: 74.6%


### Приклад використання моделі

In [16]:
pet = pd.DataFrame(
    data={
        'Type': 'Cat',
        'Age': 3,
        'Breed1': 'Tabby',
        'Gender': 'Male',
        'Color1': 'Black',
        'Color2': 'White',
        'MaturitySize': 'Small',
        'FurLength': 'Short',
        'Vaccinated': 'No',
        'Sterilized': 'No',
        'Health': 'Healthy',
        'Fee': True,
        'PhotoAmt': 2,
    },
    index=[0])

In [17]:
pet[num_cols] = kbins.transform(pet[num_cols]).astype(int).astype(str)

prob = (clf
        .predict_proba(
            scaler
            .transform(
                encoder
                .transform(
                    pet)))
        .flatten())

print(f'This pet has a {prob[1]:.1%} probability "of getting adopted"')

This pet has a 79.3% probability "of getting adopted"
